# PA4 – data preparation and unit tests
Run once. `--public` streams C4 (slow, start early); `--prefix-banks` needs the GPU.

### Setup (run every session)
Results and large artefacts go to Google Drive so a disconnected runtime loses nothing; every step is resumable.
Use an **A100** (or L4) runtime with High-RAM: *Runtime → Change runtime type*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO_URL = 'https://github.com/adilawan1/ATML-PA4.git'   # <- edit once
DRIVE = '/content/drive/MyDrive/atml-pa4'
os.makedirs(DRIVE, exist_ok=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # less fragmentation (helps on 16 GB T4)
os.environ['PA4_RESULTS'] = f'{DRIVE}/results'
os.environ['PA4_CKPT'] = f'{DRIVE}/checkpoints'
os.environ['PA4_DATA'] = f'{DRIVE}/atml-pa4-datasets'   # on Drive so data/local_source_revisions.json persists
if not os.path.exists('/content/atml-pa4'):
    !git clone -q $REPO_URL /content/atml-pa4
%cd /content/atml-pa4
!git pull -q
!pip install -q -r requirements.txt
!bash scripts/get_data.sh
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python scripts/record_env.py | tail -1

### If the runtime disconnects or times out
Nothing is lost: every result, partial result and training checkpoint is written to Google Drive as it is produced
(atomic writes, fsynced logs). Reconnect, re-run the **Setup** cell, run the progress cell below, then re-run the
cell that was interrupted: it resumes where it stopped (MQAR runs and Task-1 conditions restart from their beginning,
everything longer resumes mid-step). Keep this tab open; Colab Pro can still disconnect idle or very long sessions.

In [ ]:
!python scripts/progress.py   # what is finished / where each step will resume

In [ ]:
# Fixed public selections + tokenized windows; resolves and LOCKS Hugging Face commits on first run
!python $PA4_DATA/scripts/setup_data.py --public
# Course teacher / original-draft prefix banks for Task 3 (GPU)
!python $PA4_DATA/scripts/setup_data.py --prefix-banks
!python $PA4_DATA/scripts/check_setup.py
!cat $PA4_DATA/data/local_source_revisions.json

In [ ]:
# CPU/GPU unit tests: quantizer & packing, pruning, AWQ folding, allocation, tiled attention grads,
# MQAR mechanisms, KV byte model & cache editing, exact speculative sampler (chi^2), distillation loop
!python -m pytest -q tests